In [187]:
import importlib
import librosa 
import numpy as np 
import pandas as pd
import scipy as sci
import matplotlib.pyplot as plt
import noisereduce as nr
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import re
import parselmouth as pm
from sklearn.metrics import root_mean_squared_error
import functions
importlib.reload(functions);

In [188]:
dataset = pd.read_csv('development.csv')
sampling_rate = dataset['sampling_rate'][0]
dev_paths  = dataset['path']
y = dataset['age']
dataset = functions.get_features_form_dataset(dataset,dev_paths)


In [189]:
dataset.columns

Index(['gender', 'mean_pitch', 'max_pitch', 'min_pitch', 'jitter', 'shimmer',
       'zcr_mean', 'spectral_centroid_mean', 'hnr', 'num_pauses',
       'silence_duration', 'duration'],
      dtype='object')

In [190]:
mfccs = pd.read_csv('dev_mfccs.csv')
formants = pd.read_csv('formants_1234.csv')


In [191]:
additional_features = pd.read_csv('peak_freq_entropy_pitch_feature.csv')
all_temp = pd.read_csv('all_temporal.csv')    
# mfccsSTD = pd.read_csv('mfccsSTD.csv') non la consideriamo peggiora il modello

In [192]:
n = 13
dataset = pd.concat([dataset,mfccs.iloc[:,:n],formants,additional_features,all_temp],axis=1)



In [193]:
dataset.columns

Index(['gender', 'mean_pitch', 'max_pitch', 'min_pitch', 'jitter', 'shimmer',
       'zcr_mean', 'spectral_centroid_mean', 'hnr', 'num_pauses',
       'silence_duration', 'duration', 'mfcc-0', 'mfcc-1', 'mfcc-2', 'mfcc-3',
       'mfcc-4', 'mfcc-5', 'mfcc-6', 'mfcc-7', 'mfcc-8', 'mfcc-9', 'mfcc-10',
       'mfcc-11', 'mfcc-12', 'f1_mean', 'f2_mean', 'f3_mean', 'f4_mean',
       'f1_std', 'f2_std', 'f3_std', 'f4_std', 'peak_freq', 'peak_amplitude',
       'temporal_median', 'nVoicedFrames', 'meanAbsoluteSlope', 'nFrames',
       'entropy', 'spectralEnergy250-650', 'spectralEnergy1000-8000', 'sm',
       'sv', 'ss', 'sk', 'Time 5%', 'Time 25%', 'Time 50%', 'Time 75%',
       'Time 95%  ', 'duration_50', 'duration_90'],
      dtype='object')

In [194]:
clf = functions.try_model(dataset,y)
functions.print_feature_importance(clf,dataset)

9.643939672517401
                    feature  importance
52              duration_90    0.106302
51              duration_50    0.096467
50               Time 95%      0.061211
11                 duration    0.039919
4                    jitter    0.032577
18                   mfcc-6    0.031163
16                   mfcc-4    0.027843
28                  f4_mean    0.020701
10         silence_duration    0.017502
5                   shimmer    0.017365
44                       ss    0.017048
3                 min_pitch    0.016494
19                   mfcc-7    0.016360
22                  mfcc-10    0.016246
33                peak_freq    0.016206
46                  Time 5%    0.016042
36            nVoicedFrames    0.016013
32                   f4_std    0.015584
21                   mfcc-9    0.015423
23                  mfcc-11    0.015332
1                mean_pitch    0.015017
34           peak_amplitude    0.014939
27                  f3_mean    0.014799
37        meanAbsolute

In [195]:
eval_set = pd.read_csv('evaluation.csv')
eval_paths = eval_set['path']
eval_set = functions.get_features_form_dataset(eval_set,eval_paths,is_train=False)
eval_formants = pd.read_csv('eval_formants_1234.csv')
eval_additional_features = pd.read_csv('eval_peak_freq_entropy_pitch_feature.csv')
eval_all_temp = pd.read_csv('all_temp_features_eval.csv')
eval_mfccs = pd.read_csv('eval_mfccs.csv')
# eval_mfccsSTD = pd.read_csv('eval_mfccsSTD.csv') non consideriamo la std dei mfccs


In [196]:
eval_set = pd.concat([eval_set,eval_mfccs.iloc[:,:n],eval_formants,eval_additional_features,eval_all_temp],axis=1)

In [100]:
hist = HistGradientBoostingRegressor(warm_start=True)
x_train,x_test,y_train,y_test = train_test_split(dataset,y,test_size=0.2,random_state=42)
hist.fit(x_train,y_train)
y_pred = hist.predict(x_test)
print(root_mean_squared_error(y_test,y_pred))   

9.455097831695387


In [ ]:



hist_score = cross_val_score(HistGradientBoostingRegressor(learning_rate=0.01,max_depth=None,max_iter=1000), X, y, cv=5, scoring='neg_mean_squared_error')
hist_score

In [206]:


svr = Pipeline([ ('imputer', SimpleImputer(strategy='mean')),('scaler',StandardScaler()),('svr',SVR(C=10))])

x_train,x_test,y_train,y_test = train_test_split(dataset,y,test_size=0.2,random_state=42)
svr.fit(x_train,y_train)
y_pred = svr.predict(x_test)
print(root_mean_squared_error(y_test,y_pred))


9.674402598953154


In [198]:


randomforest = RandomForestRegressor(n_estimators=1000,min_samples_leaf= 4,min_samples_split= 2, n_jobs=-1)
hist = HistGradientBoostingRegressor(learning_rate=0.01,max_depth=None,max_iter=1000)
# VotingRegressor(randomforest, hist)
clf = VotingRegressor([('rf', randomforest), ('hist', hist),('svr',svr)])
x_train, x_test, y_train, y_test = train_test_split(dataset, y, test_size=0.2, random_state=42)
clf.fit(x_train, y_train)
ypred = clf.predict(x_test)
print(root_mean_squared_error(y_test, ypred))

9.302792283207957


In [199]:
clf = VotingRegressor([('rf', randomforest), ('hist', hist),('svr',svr)],weights=[1,1,1])
clf.fit(dataset, y) 
ypred = clf.predict(eval_set)
print(ypred)    

[33.24472267 29.68981448 23.1851994  29.60282156 32.31276546 21.04286585
 30.20390962 22.45357932 42.51626716 20.98532086 43.20363572 20.13736036
 28.61651902 20.67840185 36.32549928 29.52192872 36.57040395 26.39123945
 27.04682812 25.62208823 46.3457444  27.30486543 41.60367716 27.10885516
 30.42112211 40.2679449  24.50296622 24.70090282 37.27951861 26.35964973
 28.22838299 39.12484047 23.40941658 19.14784581 29.41658395 20.76228439
 21.40034834 34.61222095 23.27647935 22.04859763 21.94642774 20.61878842
 39.43052715 28.94269788 46.91399081 38.09112656 34.33170324 25.72263421
 21.67530012 35.21098291 37.57067601 23.97050478 28.99702077 23.09552622
 25.52103039 26.73557118 30.66433152 27.20813032 22.91677193 25.37704156
 19.96350736 19.84462171 30.6113066  21.06393182 25.87086756 26.53472031
 18.6981066  25.41849957 21.31831813 28.78683781 26.88816745 19.25939048
 26.67139625 23.50269189 30.99914125 43.58642671 32.99823512 31.72964985
 21.01409429 27.59356475 22.29079875 27.46990541 26

In [200]:
ypred = pd.DataFrame(ypred,columns=['Predicted'])
ypred.to_csv('../submitions/predicted_ansamble__rifatto_STD.csv',index=True, index_label='Id')
